# Tutorial 6.1 – Sensor Dataset Preparation and Cleaning

In the previous tutorials, you worked with physical circuits, Raspberry Pi input/output, audio, and wearable sensing. In this tutorial, we now focus only on the **sensor dataset itself**.

The main idea is simple:

> Before fuzzy logic can be applied, the raw EmotiBit CSV files must be inspected, cleaned, synchronized, and saved as one usable sensor table.

In this notebook, you will load selected EmotiBit CSV files, inspect their structure, handle different sampling rates, remove a small startup/outlier issue, compute movement intensity, and save one clean table for the next notebook.

The fuzzy logic and linguistic variables are continued in **Tutorial 6.2**.


## Learning objectives

By the end of this tutorial, you should be able to:

1. inspect selected EmotiBit sensor CSV files;
2. understand why different sensor files have different sampling rates;
3. prepare a clean time-based sensor table with one row per second;
4. compute a simple movement-intensity value from accelerometer axes;
5. save the cleaned dataset so that it can be used in Tutorial 6.2.


## Prerequisites

This notebook assumes that you already have an EmotiBit recording exported into individual CSV files. For this first version, we use only the selected files below:

| Signal | File | Meaning |
|---|---|---|
| `HR` | `2026-04-26_22-34-20-119626_HR.csv` | Heart rate in beats per minute |
| `T1` | `2026-04-26_22-34-20-119626_T1.csv` | Skin/contact temperature |
| `EA` | `2026-04-26_22-34-20-119626_EA.csv` | Electrodermal activity |
| `SF` | `2026-04-26_22-34-20-119626_SF.csv` | Skin conductance response frequency |
| `AX`, `AY`, `AZ` | accelerometer CSV files | Movement components along three axes |

Place these files in a folder named `data` next to this notebook.

**Before running the notebook, install the required Python packages in the same environment where JupyterLab is running:**

```bash
pip install pandas numpy duckdb matplotlib jupyterlab nbconvert
```

For Tutorial 6.1, the important packages are mainly `pandas` and `numpy`. `duckdb` is used in Tutorial 6.2.

The JSON metadata file is not required for this first version.


## General workflow

The workflow of this notebook is:

```text
Selected EmotiBit CSV files
        ↓
Load and inspect in Python
        ↓
Create one clean sensor table per second
        ↓
Save clean table to tutorial_data/processed/clean_emotibit_readings.csv
        ↓
Use this clean table in Tutorial 6.2
```

The goal of this notebook is to prepare the dataset properly before applying fuzzy logic.

The cleaned dataset is stored in the shared `tutorial_data` folder so that Tutorial 6.2 can load it automatically later without manual file copying.


In [7]:
# If needed, install the required packages in your virtual environment first.
# Run this in Terminal, not inside the notebook, if packages are missing:
# pip install pandas numpy duckdb matplotlib jupyterlab nbconvert

from pathlib import Path
import math
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# The notebook can work with the current local development layout
# and with the final CWW-Tutorials repository layout.
candidate_data_dirs = [
    Path("data"),
    Path("Tutorial 6.1") / "data",
    Path("tutorial_data") / "raw",
]

DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), candidate_data_dirs[0])

# Shared course-data location used across Tutorials 5.2, 6.1 and 6.2.
PROCESSED_DIR = Path("..") / "tutorial_data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = {
    "HR": DATA_DIR / "2026-04-26_22-34-20-119626_HR.csv",
    "T1": DATA_DIR / "2026-04-26_22-34-20-119626_T1.csv",
    "EA": DATA_DIR / "2026-04-26_22-34-20-119626_EA.csv",
    "SF": DATA_DIR / "2026-04-26_22-34-20-119626_SF.csv",
    "AX": DATA_DIR / "2026-04-26_22-34-20-119626_AX.csv",
    "AY": DATA_DIR / "2026-04-26_22-34-20-119626_AY.csv",
    "AZ": DATA_DIR / "2026-04-26_22-34-20-119626_AZ.csv",
}

missing = [str(path) for path in required_files.values() if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required CSV file(s):\n" + "\n".join(missing)
    )

print(f"Using sensor CSV folder: {DATA_DIR.resolve()}")
print("All required CSV files were found.")


Using sensor CSV folder: /Users/patanimrankhan/Desktop/CWW-Tutorials/Tutorial 6.1/data
All required CSV files were found.


## Part A — Load and inspect the selected CSV files

Each individual EmotiBit CSV file has a similar structure. The last column usually contains the actual sensor value, for example `HR` for heart rate or `T1` for temperature.

In [8]:
def load_signal(tag: str) -> pd.DataFrame:
    # Load one selected EmotiBit signal CSV and standardize its timestamp.
    path = required_files[tag]
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["LocalTimestamp"], unit="s")
    return df

signals = {tag: load_signal(tag) for tag in required_files}

summary_rows = []
for tag, df in signals.items():
    value_col = tag
    summary_rows.append({
        "signal": tag,
        "rows": len(df),
        "start_time": df["datetime"].min(),
        "end_time": df["datetime"].max(),
        "min": df[value_col].min(),
        "mean": df[value_col].mean(),
        "max": df[value_col].max(),
    })

pd.DataFrame(summary_rows)

,signal,rows,start_time,end_time,min,mean,max
0,HR,133,2026-04-26 20:34:24.000771046,2026-04-26 20:36:40.696666002,37.250000,78.324286,133.150000
1,T1,1041,2026-04-26 20:34:21.258713007,2026-04-26 20:36:44.847754002,3.000000,35.711064,36.073000
2,EA,2082,2026-04-26 20:34:21.265712976,2026-04-26 20:36:44.854753017,0.030253,0.030265,0.030325
3,SF,417,2026-04-26 20:34:21.465718031,2026-04-26 20:36:44.721750975,0.006700,0.187118,0.998100
4,AX,3488,2026-04-26 20:34:21.267714024,2026-04-26 20:36:44.828752995,-1.870000,0.283299,2.618000
5,AY,3488,2026-04-26 20:34:21.267714024,2026-04-26 20:36:44.828752995,-3.703000,-0.608511,1.978000
6,AZ,3488,2026-04-26 20:34:21.267714024,2026-04-26 20:36:44.828752995,-1.520000,0.633365,2.640000


In [9]:
# Look at one example file.
signals["HR"].head()

,LocalTimestamp,EmotiBitTimestamp,PacketNumber,DataLength,TypeTag,ProtocolVersion,DataReliability,HR,datetime
0,1.777236e+09,185674.0,29692,1,HR,1,100,54.01,2026-04-26 20:34:24.000771046
1,1.777236e+09,186074.0,29764,1,HR,1,100,75.34,2026-04-26 20:34:24.400779963
2,1.777236e+09,186594.0,29852,1,HR,1,100,84.24,2026-04-26 20:34:24.920790911
3,1.777236e+09,186954.0,29925,1,HR,1,100,102.56,2026-04-26 20:34:25.280797958
4,1.777236e+09,188527.0,30189,1,HR,1,100,88.31,2026-04-26 20:34:26.853832006


## Part B — Prepare one clean sensor table

The selected sensor files have different sampling rates. For example, accelerometer values appear more frequently than heart rate values. To make this tutorial simple, we aggregate the values into **one row per second**.

The clean table will contain:

- `timestamp`
- `heart_rate`
- `skin_temperature`
- `eda`
- `scr_frequency`
- `ax`, `ay`, `az`
- `accel_magnitude`
- `movement_intensity`

For movement, we compute the magnitude of the acceleration vector:

\[
\sqrt{AX^2 + AY^2 + AZ^2}
\]

Then we estimate movement intensity as the distance from approximately `1g`, because a still accelerometer often measures around one gravity unit.

In [10]:
def per_second_mean(tag: str, value_name: str | None = None) -> pd.DataFrame:
    # Return one mean value per second for a signal.
    value_name = value_name or tag
    df = signals[tag][["datetime", tag]].copy()
    df = df.set_index("datetime")
    result = df.resample("1s").mean(numeric_only=True)
    result = result.rename(columns={tag: value_name})
    return result

hr = per_second_mean("HR", "heart_rate")
temp = per_second_mean("T1", "skin_temperature")
eda = per_second_mean("EA", "eda")
sf = per_second_mean("SF", "scr_frequency")

# Clean one obvious temperature startup/outlier issue.
# For this dataset, valid skin/contact temperature values are expected to be well above room-temperature startup artefacts.
temp.loc[temp["skin_temperature"] < 25, "skin_temperature"] = np.nan

# Accelerometer axes share the same timestamps in this export.
ax = signals["AX"][["datetime", "AX"]].set_index("datetime").rename(columns={"AX": "ax"})
ay = signals["AY"][["datetime", "AY"]].set_index("datetime").rename(columns={"AY": "ay"})
az = signals["AZ"][["datetime", "AZ"]].set_index("datetime").rename(columns={"AZ": "az"})
accel = ax.join([ay, az], how="inner")
accel["accel_magnitude"] = np.sqrt(accel["ax"]**2 + accel["ay"]**2 + accel["az"]**2)
accel["movement_intensity"] = (accel["accel_magnitude"] - 1.0).abs()
accel_1s = accel.resample("1s").mean(numeric_only=True)

clean = hr.join([temp, eda, sf, accel_1s], how="outer").sort_index()
clean = clean.interpolate(method="time", limit_direction="both")
clean = clean.reset_index().rename(columns={"datetime": "timestamp"})

# Keep a stable order of columns.
clean = clean[[
    "timestamp", "heart_rate", "skin_temperature", "eda", "scr_frequency",
    "ax", "ay", "az", "accel_magnitude", "movement_intensity"
]]

clean.head()

,timestamp,heart_rate,skin_temperature,eda,scr_frequency,ax,ay,az,accel_magnitude,movement_intensity
0,2026-04-26 20:34:21,71.196667,35.694333,0.030260,0.992350,0.298933,-0.375956,0.886178,1.007977,0.007977
1,2026-04-26 20:34:22,71.196667,35.679500,0.030260,0.963933,0.299360,-0.375320,0.885320,1.007267,0.007550
2,2026-04-26 20:34:23,71.196667,35.664000,0.030260,0.930867,0.300480,-0.372920,0.886240,1.007533,0.009006
3,2026-04-26 20:34:24,71.196667,35.702625,0.030259,0.898933,0.299520,-0.377680,0.885200,1.007984,0.008095
4,2026-04-26 20:34:25,102.560000,35.716286,0.030259,0.868100,0.312538,-0.371000,0.881346,1.006098,0.007859


In [11]:
clean.describe(include="all")

,timestamp,heart_rate,skin_temperature,eda,scr_frequency,ax,ay,az,accel_magnitude,movement_intensity
count,144,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000
mean,2026-04-26 20:35:32.500000,72.245694,35.744230,0.030265,0.200893,0.281412,-0.595902,0.644736,1.162640,0.300151
min,2026-04-26 20:34:21,37.250000,35.490625,0.030255,0.006800,-0.728520,-1.657880,-0.113360,0.917582,0.000725
25%,2026-04-26 20:34:56.750000128,60.227500,35.640625,0.030256,0.023625,0.080220,-0.932620,0.404670,0.993529,0.017795
50%,2026-04-26 20:35:32.500000,70.060000,35.771357,0.030258,0.082300,0.260720,-0.399890,0.736540,1.018979,0.052784
75%,2026-04-26 20:36:08.249999872,79.847500,35.861286,0.030267,0.286625,0.494900,-0.246572,0.894930,1.312476,0.631290
max,2026-04-26 20:36:44,122.096667,35.957250,0.030317,0.992350,0.862040,-0.083200,1.086840,1.876829,1.002734
std,NaN,17.648037,0.130867,0.000014,0.250631,0.278086,0.437661,0.304448,0.255207,0.339274


In [12]:
clean_csv_path = PROCESSED_DIR / "clean_emotibit_readings.csv"
clean.to_csv(clean_csv_path, index=False)
print(f"Clean sensor table saved to: {clean_csv_path}")
print(f"Rows: {len(clean)}")


Clean sensor table saved to: ../tutorial_data/processed/clean_emotibit_readings.csv
Rows: 144


## Output for Tutorial 6.2

At the end of this notebook, the cleaned dataset is saved in the shared course-data folder:

```text
tutorial_data/
└── processed/
    └── clean_emotibit_readings.csv
```

Open **Tutorial 6.2** next. It will load this cleaned CSV file automatically and apply DuckDB + fuzzy linguistic variables.


## Troubleshooting

### 1. `FileNotFoundError` for CSV files

Check that the selected CSV files are available in one of the supported locations.

Recommended final structure:

```text
Tutorial 6.1/
├── Tutorial 6.1 - Sensor Dataset Preparation and Cleaning.ipynb
└── data/
    ├── 2026-04-26_22-34-20-119626_HR.csv
    ├── 2026-04-26_22-34-20-119626_T1.csv
    ├── 2026-04-26_22-34-20-119626_EA.csv
    ├── 2026-04-26_22-34-20-119626_SF.csv
    ├── 2026-04-26_22-34-20-119626_AX.csv
    ├── 2026-04-26_22-34-20-119626_AY.csv
    └── 2026-04-26_22-34-20-119626_AZ.csv
```

The notebook also supports `data/` in the current Jupyter working directory and `tutorial_data/raw/`.

### 2. `ModuleNotFoundError`

Install the required Python packages inside your virtual environment:

```bash
pip install pandas numpy duckdb matplotlib jupyterlab nbconvert
```

Make sure the package installation happens in the same environment where JupyterLab is running.

### 3. The clean CSV file is not created

Run all cells in this notebook from top to bottom. The cleaned dataset is created at:

```text
tutorial_data/processed/clean_emotibit_readings.csv
```


## Summary

In this tutorial, you moved from individual EmotiBit CSV files to one cleaned sensor table.

You created:

- a summary of the selected sensor CSV files;
- a one-row-per-second table;
- a cleaned skin/contact temperature column;
- a movement-intensity value from accelerometer data;
- a shared processed dataset at `tutorial_data/processed/clean_emotibit_readings.csv`.

This cleaned file becomes the input for **Tutorial 6.2**, where fuzzy linguistic variables are created and later compared with the linguistic self-report recorded in Tutorial 5.2.


## References

- EmotiBit exported CSV sensor files used in this tutorial.
- Donald J. Norris, *Beginning Artificial Intelligence with the Raspberry Pi*, especially the fuzzy logic system chapter.
- Daniel Fasel, *Fuzzy Data Warehousing for Performance Measurement: Concept and Implementation*.
- Michael Kaufmann and Andreas Meier, *SQL and NoSQL Databases*, fuzzy databases section.
- DuckDB documentation for SQL, views, and macros.
